In [1]:
# ✅ Step 1: Imports
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document
from dotenv import load_dotenv
import os

# ✅ Step 2: Load API key
load_dotenv(dotenv_path=".env")
openai_api_key = os.getenv("OPENAI_API_KEY")

# ✅ Step 3: Setup LLM and Embeddings
llm = ChatOpenAI(temperature=0, model="gpt-4o-mini", api_key=openai_api_key)
embedding = OpenAIEmbeddings(api_key=openai_api_key)

# ✅ Step 4: Load and split document
with open("sample.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

splitter = CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)
texts = splitter.split_text(raw_text)
documents = [Document(page_content=t) for t in texts]

# ✅ Step 5: Vector Store (optional for Retriever chains)
vectorstore = FAISS.from_texts(texts, embedding)
retriever = vectorstore.as_retriever()

print("✅ Base setup complete.")

C:\Users\xsanthkum\AppData\Local\Temp\ipykernel_38440\3603342034.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


✅ Base setup complete.


In [2]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "Use the following context to answer the question:\n\n{context}\n\nQuestion: {question}"
)

stuff_chain = create_stuff_documents_chain(
    llm,
    prompt,
    document_variable_name="context"
)

response = stuff_chain.invoke({
    "context": documents[:3],
    "question": "What is this document about?"
})

print("\n📌 StuffDocumentsChain Answer:", response)


📌 StuffDocumentsChain Answer: This document is about LangChain, a framework designed for building applications that utilize large language models (LLMs). It provides information on its creator, Harrison Chase, and highlights the features and common use cases of LangChain, such as supporting retrieval-augmented generation (RAG), agents, memory, tools, and its applications in chatbots, document question-and-answer systems, and AI workflows.


In [4]:
from IPython.display import display, HTML

innerHtml = """
<div style="
    font-family:Segoe UI, Arial;
    max-width:1200px;
    margin:auto;
    padding:20px;
    background:#f4f6f9;
    border-radius:15px;
">

<h1 style="color:#4F46E5;">
📚 LangChain QA Chains Comparison
</h1>

<p>
The table below summarizes commonly used Retrieval and QA chains, whether they return citations/sources,
their output format, and usage recommendations.
</p>

<table style="
    width:100%;
    border-collapse:collapse;
    background:white;
">

<tr style="background:#4F46E5;color:white;">
    <th style="padding:12px;">Chain / Retriever</th>
    <th style="padding:12px;">Returns Citations</th>
    <th style="padding:12px;">Output Keys</th>
    <th style="padding:12px;">Status</th>
    <th style="padding:12px;">Best Use Case</th>
</tr>

<tr>
    <td style="padding:10px;"><b>RetrievalQAWithSourcesChain</b></td>
    <td style="padding:10px;">✅ Yes</td>
    <td style="padding:10px;">answer, sources</td>
    <td style="padding:10px;">✅ Stable</td>
    <td style="padding:10px;">RAG applications requiring source references.</td>
</tr>

<tr style="background:#f8f9fc;">
    <td style="padding:10px;"><b>ConversationalRetrievalChain</b></td>
    <td style="padding:10px;">✅ Yes</td>
    <td style="padding:10px;">answer, source_documents</td>
    <td style="padding:10px;">✅ Stable</td>
    <td style="padding:10px;">Chatbots with memory and document retrieval.</td>
</tr>

<tr>
    <td style="padding:10px;"><b>MultiRetrievalQAChain</b></td>
    <td style="padding:10px;">✅ Yes</td>
    <td style="padding:10px;">Depends on sub-chains</td>
    <td style="padding:10px;">✅ Stable</td>
    <td style="padding:10px;">Multiple knowledge sources and routers.</td>
</tr>

<tr style="background:#f8f9fc;">
    <td style="padding:10px;"><b>VectorDBQAWithSourcesChain</b></td>
    <td style="padding:10px;">✅ Yes</td>
    <td style="padding:10px;">answer, sources</td>
    <td style="padding:10px;">⚠️ Legacy</td>
    <td style="padding:10px;">Older VectorDB-based RAG implementations.</td>
</tr>

<tr>
    <td style="padding:10px;"><b>Tool using RetrievalQAWithSources</b></td>
    <td style="padding:10px;">✅ Yes</td>
    <td style="padding:10px;">answer, sources</td>
    <td style="padding:10px;">✅ Agent Compatible</td>
    <td style="padding:10px;">Agents that need document-backed answers.</td>
</tr>

<tr style="background:#f8f9fc;">
    <td style="padding:10px;"><b>RetrievalQA</b></td>
    <td style="padding:10px;">❌ No</td>
    <td style="padding:10px;">answer only</td>
    <td style="padding:10px;">✅ Stable</td>
    <td style="padding:10px;">Simple RAG without source citations.</td>
</tr>

<tr>
    <td style="padding:10px;"><b>RefineDocumentsChain</b></td>
    <td style="padding:10px;">❌ No</td>
    <td style="padding:10px;">answer only</td>
    <td style="padding:10px;">✅ Stable</td>
    <td style="padding:10px;">Progressively refine answers from multiple documents.</td>
</tr>

<tr style="background:#f8f9fc;">
    <td style="padding:10px;"><b>StuffDocumentsChain</b></td>
    <td style="padding:10px;">❌ No</td>
    <td style="padding:10px;">answer only</td>
    <td style="padding:10px;">✅ Stable</td>
    <td style="padding:10px;">Small document sets that fit entirely into context.</td>
</tr>

</table>

<br>

<div style="
background:#DCFCE7;
padding:20px;
border-radius:10px;
">

<h2>🎯 Recommendation Matrix</h2>

<table style="width:100%;border-collapse:collapse;background:white;">

<tr style="background:#22C55E;color:white;">
    <th style="padding:10px;">Scenario</th>
    <th style="padding:10px;">Recommended Chain</th>
</tr>

<tr>
    <td style="padding:10px;">Need Answers with Source References</td>
    <td style="padding:10px;">✅ RetrievalQAWithSourcesChain</td>
</tr>

<tr>
    <td style="padding:10px;">Chatbot with Memory + RAG</td>
    <td style="padding:10px;">✅ ConversationalRetrievalChain</td>
</tr>

<tr>
    <td style="padding:10px;">Multiple Knowledge Bases</td>
    <td style="padding:10px;">✅ MultiRetrievalQAChain</td>
</tr>

<tr>
    <td style="padding:10px;">Agent-Based RAG</td>
    <td style="padding:10px;">✅ RetrievalQAWithSources Tool</td>
</tr>

<tr>
    <td style="padding:10px;">Simple Document QA</td>
    <td style="padding:10px;">✅ RetrievalQA</td>
</tr>

<tr>
    <td style="padding:10px;">Large Documents Requiring Refinement</td>
    <td style="padding:10px;">✅ RefineDocumentsChain</td>
</tr>

<tr>
    <td style="padding:10px;">Small PDFs or Small KB</td>
    <td style="padding:10px;">✅ StuffDocumentsChain</td>
</tr>

</table>

</div>

<br>

<div style="
background:#FEF3C7;
padding:20px;
border-radius:10px;
">

<h2>💡 Easy Way to Remember</h2>

<ul>
<li><b>RetrievalQA</b> → Answer Only</li>
<li><b>RetrievalQAWithSources</b> → Answer + Sources</li>
<li><b>ConversationalRetrievalChain</b> → Chat + Memory + Sources</li>
<li><b>MultiRetrievalQAChain</b> → Multiple Knowledge Bases</li>
<li><b>StuffDocumentsChain</b> → Small Documents</li>
<li><b>RefineDocumentsChain</b> → Large Documents</li>
<li><b>RetrievalQA Tool</b> → Agent + RAG + Sources</li>
</ul>

</div>

</div>
"""

display(HTML(innerHtml))

In [3]:
# Initial prompt to summarize the first chunk
# Use case: Starts with a base answer and refines it using subsequent documents — good when each chunk contributes incrementally.

from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser

initial_prompt = PromptTemplate.from_template("""
Write a concise summary of the following text:

{context}
""")

# Refine prompt to update the previous summary with new context
refine_prompt = PromptTemplate.from_template("""
We have an existing summary:
"{existing_answer}"

Refine the summary with this new context:
"{context}"

If the context isn't useful, return the original summary.
""")

# Set up individual chains
initial_summary_chain = initial_prompt | llm | StrOutputParser()
refine_summary_chain = refine_prompt | llm | StrOutputParser()

# Start with first chunk
summary = initial_summary_chain.invoke({"context": documents[0].page_content})

# Iteratively refine with remaining docs
for doc in documents[1:]:
    summary = refine_summary_chain.invoke({
        "existing_answer": summary,
        "context": doc.page_content
    })

# Output final summary
print("📄 Refined Summary:\n")
print(summary)

📄 Refined Summary:

LangChain is a framework developed by Harrison Chase for building applications using large language models (LLMs). It supports various features such as retrieval-augmented generation (RAG), agents, memory, and tools, and is commonly utilized in chatbots, document Q&A, and AI workflows.
